# Task 09: Reference-Based Color Standardization with Histogram Matching

Nesta atividade, você padronizará a cor de uma imagem RGB alvo com base em uma imagem RGB de referência. O matching aproxima, canal a canal, a distribuição do alvo da distribuição de referência.

## Roteiro

1. Gerar uma cena RGB sintética e suas versões de referência e alvo.
2. Inspecionar histogramas RGB antes da correção.
3. Calcular histogramas e CDFs.
4. Construir uma LUT por canal.
5. Aplicar histogram matching RGB.
6. Comparar imagens, histogramas e CDFs.

## 1. Importações

Use apenas NumPy e Matplotlib. Não use implementações prontas de histogram matching ou padronização automática de cor.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## 2. Reference e Source

A cena base possui gradientes, regiões e detalhes. A referência recebe maior ganho no canal R e menor no B, criando uma aparência mais quente. A imagem source recebe os ganhos inversos, criando uma dominante azulada.

In [ ]:
height, width = 128, 160
y, x = np.mgrid[:height, :width]
scene = np.empty((height, width, 3), dtype=np.float64)
scene[..., 0] = 40 + 1.1 * x + 0.2 * y
scene[..., 1] = 30 + 0.4 * x + 0.8 * y
scene[..., 2] = 50 + 0.3 * x + 0.9 * y
scene[30:90, 45:100] = (170, 110, 70)
scene[(x - 120) ** 2 + (y - 35) ** 2 < 18**2] = (70, 180, 130)

reference_image = np.clip(scene * np.array([1.20, 1.00, 0.75]), 0, 255).astype(np.uint8)
source_image = np.clip(scene * np.array([0.75, 1.00, 1.20]), 0, 255).astype(np.uint8)


def show_images(items):
    figure, axes = plt.subplots(1, len(items), figsize=(5 * len(items), 4))
    for axis, (image, title) in zip(axes, items, strict=True):
        axis.imshow(image)
        axis.set_title(title)
        axis.axis("off")
    figure.tight_layout()


show_images(
    [
        (reference_image, "Reference: mais quente"),
        (source_image, "Source: mais azulada"),
    ]
)

## 3. Histogramas RGB antes da correção

O código de visualização está pronto. Cada cor de linha representa o canal RGB correspondente, somente para facilitar a leitura do gráfico.

In [ ]:
def plot_rgb_histograms(images):
    figure, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 4))
    colors = ("red", "green", "blue")
    labels = ("R", "G", "B")
    for axis, (image, title) in zip(axes, images, strict=True):
        for channel, (color, label) in enumerate(zip(colors, labels, strict=True)):
            histogram = np.bincount(image[..., channel].ravel(), minlength=256)
            axis.plot(histogram, color=color, label=label)
        axis.set_title(title)
        axis.set_xlim(0, 255)
        axis.set_xlabel("Intensity")
        axis.set_ylabel("Count")
        axis.legend()
    figure.tight_layout()


plot_rgb_histograms(
    [(reference_image, "Reference histograms"), (source_image, "Source histograms")]
)

## 4. Histograma e CDF

Implemente `compute_histogram_cdf`. A entrada é um canal 2D `uint8`. O histograma possui exatamente 256 bins, e a CDF normalizada é calculada a partir do histograma. A CDF deve ter dtype `float64`, valores entre `0.0` e `1.0` e terminar em `1.0`.

In [ ]:
def compute_histogram_cdf(channel):
    """Computes the 256-bin histogram and normalized CDF of a uint8 channel."""
    histogram = None
    cdf = None

    ### START CODE HERE ###

    # TODO

    ### END CODE HERE ###

    return histogram, cdf

## 5. Matching de um canal

Implemente `match_channel`. Calcule as CDFs source e reference, crie uma LUT de 256 posições e aplique a LUT ao canal source. Para cada intensidade `i`, selecione a menor intensidade `j` da referência que satisfaz `reference_cdf[j] >= source_cdf[i]`. `np.searchsorted` pode ser usado. Preserve as entradas e retorne um novo canal `uint8`.

In [ ]:
def match_channel(source_channel, reference_channel):
    """Matches the histogram of one source channel to a reference channel."""
    matched = None

    ### START CODE HERE ###

    # TODO

    ### END CODE HERE ###

    return matched

## 6. Matching RGB

Implemente `match_color_histograms`. As entradas são arrays RGB `uint8` com shape `(H, W, 3)`. As dimensões espaciais podem ser diferentes. Aplique `match_channel` independentemente em `source[..., 0]`, `source[..., 1]` e `source[..., 2]`, usando o canal correspondente da referência. Rejeite entradas que não sejam RGB ou não sejam `uint8`.

In [ ]:
def match_color_histograms(source, reference):
    """Matches the RGB channel histograms of source to reference."""
    matched = None

    ### START CODE HERE ###

    # TODO

    ### END CODE HERE ###

    return matched

## 7. Teste determinístico

Execute depois de implementar as três funções. O teste verifica histograma, CDF, LUT, matching RGB, imutabilidade, dtype, shape e a possibilidade de usar dimensões espaciais diferentes.

In [ ]:
channel = np.array([[0, 0], [1, 3]], dtype=np.uint8)
histogram, cdf = compute_histogram_cdf(channel)
assert histogram.shape == (256,)
assert cdf.shape == (256,)
assert (
    histogram[0] == 2 and histogram[1] == 1 and histogram[2] == 0 and histogram[3] == 1
)
assert cdf.dtype == np.float64
assert cdf[0] == 0.5 and cdf[1] == 0.75 and cdf[2] == 0.75
assert cdf[3] == 1.0 and cdf[255] == 1.0

source_channel = np.array([[0, 0], [1, 1]], dtype=np.uint8)
reference_channel = np.array([[10, 10], [20, 20]], dtype=np.uint8)
expected_channel = np.array([[10, 10], [20, 20]], dtype=np.uint8)
matched_channel = match_channel(source_channel, reference_channel)
assert np.array_equal(matched_channel, expected_channel)

source_rgb = np.array(
    [[[0, 10, 100], [0, 10, 100]], [[1, 20, 200], [1, 20, 200]]], dtype=np.uint8
)
reference_rgb = np.array(
    [[[50, 30, 20], [50, 30, 20]], [[100, 60, 40], [100, 60, 40]]], dtype=np.uint8
)
expected_rgb = reference_rgb.copy()
source_original = source_rgb.copy()
reference_original = reference_rgb.copy()
matched_rgb = match_color_histograms(source_rgb, reference_rgb.reshape(1, 4, 3))
assert np.array_equal(matched_rgb, expected_rgb)
assert matched_rgb.shape == source_rgb.shape and matched_rgb.dtype == np.uint8
assert matched_rgb is not source_rgb
assert np.array_equal(source_rgb, source_original)
assert np.array_equal(reference_rgb, reference_original)

for invalid in (
    np.zeros((2, 2), dtype=np.uint8),
    np.zeros((2, 2, 3), dtype=np.float32),
):
    try:
        match_color_histograms(invalid, reference_rgb)
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError("Invalid source must be rejected.")

print("Test passed!")

## 8. Aplicação e comparação visual

Depois de completar o matching RGB, aplique-o à imagem source e compare as três versões lado a lado.

In [ ]:
matched_image = match_color_histograms(source_image, reference_image)
show_images(
    [
        (reference_image, "Reference"),
        (source_image, "Source"),
        (matched_image, "Matched"),
    ]
)

## 9. Histogramas e CDFs depois da correção

Observe como cada histograma e CDF da imagem matched se aproxima do canal correspondente da referência.

In [ ]:
plot_rgb_histograms(
    [
        (reference_image, "Reference histograms"),
        (source_image, "Source histograms"),
        (matched_image, "Matched histograms"),
    ]
)


def plot_rgb_cdfs(images):
    figure, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 4))
    colors = ("red", "green", "blue")
    labels = ("R", "G", "B")
    for axis, (image, title) in zip(axes, images, strict=True):
        for channel, (color, label) in enumerate(zip(colors, labels, strict=True)):
            _, cdf = compute_histogram_cdf(image[..., channel])
            axis.plot(cdf, color=color, label=label)
        axis.set_title(title)
        axis.set_xlim(0, 255)
        axis.set_ylim(0, 1)
        axis.set_xlabel("Intensity")
        axis.set_ylabel("CDF")
        axis.legend()
    figure.tight_layout()


plot_rgb_cdfs(
    [
        (reference_image, "Reference CDFs"),
        (source_image, "Source CDFs"),
        (matched_image, "Matched CDFs"),
    ]
)

## Explore

Observe a dominante principal da imagem source, como os histogramas R, G e B mudam e se a imagem corrigida se aproxima visualmente da referência. Explique por que histogram matching é diferente de equalizar cada imagem separadamente e por que uma referência pode ser útil em uma linha de produção.